# Fast and Safe - Validación del DW
Verifica que todas las tablas quedaron bien cargadas y responde las preguntas del enunciado.

In [58]:
import pandas as pd
import yaml
from sqlalchemy import create_engine

with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

cfg = config['TARGET_DB']
url = f"{cfg['drivername']}://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
dw = create_engine(url)
print('Conexión al DW OK')

Conexión al DW OK


## 1. Conteo de filas por tabla

In [41]:
tablas = [
    'dim_fecha', 'dim_cliente', 'dim_sede', 'dim_mensajero',
    'dim_ciudad', 'dim_tipo_servicio', 'dim_estado_servicio',
    'dim_tipo_novedad', 'hecho_servicio', 'hecho_transiciones_estado',
    'resumen_novedades'
]
for t in tablas:
    try:
        n = pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', dw).iloc[0,0]
        print(f'  {t:35s}: {n} filas')
    except Exception as e:
        print(f'  {t:35s}: ERROR - {e}')

  dim_fecha                          : 333 filas
  dim_cliente                        : 28 filas
  dim_sede                           : 53 filas
  dim_mensajero                      : 51 filas
  dim_ciudad                         : 20 filas
  dim_tipo_servicio                  : 5 filas
  dim_estado_servicio                : 7 filas
  dim_tipo_novedad                   : 2 filas
  hecho_servicio                     : 28328 filas
  hecho_transiciones_estado          : 63180 filas
  resumen_novedades                  : 2 filas


## 2. Vista previa dimensiones

In [59]:
pd.read_sql('SELECT * FROM dim_fecha LIMIT 5', dw)

,key_dim_fecha,fecha,year,month,day,hour,weekday,quarter,is_holiday,nombre_festivo,nombre_mes,nombre_dia,es_fin_semana
0,1,2023-09-19,2023,9,19,0,1,3,False,DESCONOCIDO,Septiembre,Martes,False
1,2,2023-09-20,2023,9,20,0,2,3,False,DESCONOCIDO,Septiembre,Miã©rcoles,False
2,3,2023-09-21,2023,9,21,0,3,3,False,DESCONOCIDO,Septiembre,Jueves,False
3,4,2023-09-22,2023,9,22,0,4,3,False,DESCONOCIDO,Septiembre,Viernes,False
4,5,2023-09-23,2023,9,23,0,5,3,False,DESCONOCIDO,Septiembre,Sã¡bado,True


In [43]:
pd.read_sql('SELECT * FROM dim_cliente', dw)

,key_dim_cliente,cod_cliente,nit_cliente,nombre_cliente,sector_economico
0,1,1,25,Cliente 2,S
1,2,2,123,Cliente 1,industrial
2,3,6,24390-3,CLINICA DEPORTIVA DEL SUR,salud
3,4,19,8301821,HOSPITAL ORTOPEDICO DE COLOMBIA,salud
4,5,8,5017350-8,CLINICA NEFROLOGOS DE CALI,salud
5,6,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE,salud
6,7,4,306215-0,CRUZ AZUL-LIFE,salud
7,8,5,300513-3,CLINICA CALI -JOVEN,salud
8,9,7,951033-8,CLINICA COFFE -HEALTH,salud
9,10,9,149757-6,UNIDAD DE TRAUMA DEL OESTE,salud


In [60]:
pd.read_sql('SELECT * FROM dim_mensajero', dw)

,key_dim_mensajero,cod_mensajero,nombre,ciudad_base,tipo_vehiculo
0,1,84,pepito_el_rapido pepito_el_furioso,CALI,Camioneta
1,2,1,pepito_el_rapido pepito_el_furioso,ACOPI YUMBO,Camioneta
2,3,15,pepito_el_rapido pepito_el_furioso,CALI,Camioneta
3,4,24,pepito_el_rapido pepito_el_furioso,CALI,Moto
4,5,32,pepito_el_rapido pepito_el_furioso,CALI,Moto
5,6,42,pepito_el_rapido pepito_el_furioso,CALI,Moto
6,7,28,pepito_el_rapido pepito_el_furioso,CALI,Moto
7,8,47,pepito_el_rapido pepito_el_furioso,CALI,Moto
8,9,83,pepito_el_rapido pepito_el_furioso,CALI,Moto
9,10,30,pepito_el_rapido pepito_el_furioso,CALI,Moto


In [64]:
pd.read_sql('SELECT * FROM dim_estado_servicio ORDER BY orden_estado', dw)

,key_dim_estado,cod_estado,nombre_estado,orden_estado,es_estado_final
0,1,1,Iniciado,1,False
1,2,2,Con mensajero Asignado,2,False
2,3,3,Con novedad,3,False
3,4,4,Recogido por mensajero,4,False
4,5,5,Entregado en destino,5,True
5,6,6,Terminado completo,6,True
6,0,0,DESCONOCIDO,99,False


In [65]:
pd.read_sql('SELECT * FROM dim_tipo_novedad', dw)

,key_dim_tipo_novedad,cod_tipo_novedad,nombre,categoria
0,17,2,No puedo continuar,OTRO
1,18,1,Novedades del servicio,OTRO


## 3. Preguntas del enunciado
### P1 - ¿En qué meses del año los clientes solicitan más servicios?

In [67]:
pd.read_sql("""
    SELECT f.month, f.nombre_mes, COUNT(*) AS total_servicios
    FROM hecho_servicio h
    JOIN dim_fecha f ON f.key_dim_fecha = h.fk_fecha_solicitud
    WHERE h.fk_fecha_solicitud != 0
    GROUP BY f.month, f.nombre_mes
    ORDER BY total_servicios DESC
""", dw)

,month,nombre_mes,total_servicios
0,5,Mayo,4724
1,7,Julio,4550
2,4,Abril,4473
3,8,Agosto,4303
4,6,Junio,4184
5,3,Marzo,3319
6,2,Febrero,2466
7,1,Enero,286
8,12,Diciembre,12
9,11,Noviembre,9


### P2 - ¿Cuáles son los días con más solicitudes?

In [69]:
pd.read_sql("""
    SELECT f.weekday, f.nombre_dia, COUNT(*) AS total
    FROM hecho_servicio h
    JOIN dim_fecha f ON f.key_dim_fecha = h.fk_fecha_solicitud
    WHERE h.fk_fecha_solicitud != 0
    GROUP BY f.weekday, f.nombre_dia
    ORDER BY f.weekday
""", dw)

,weekday,nombre_dia,total
0,0,Lunes,4289
1,1,Martes,5368
2,2,Miã©rcoles,4956
3,3,Jueves,5142
4,4,Viernes,5272
5,5,Sã¡bado,2466
6,6,Domingo,835


### P3 - hora que los mensajeros están más ocupados.

In [71]:
pd.read_sql("""
    SELECT
    EXTRACT(HOUR FROM timestamp_estado) AS hora,
    m.nombre                            AS mensajero,
    COUNT(DISTINCT id_servicio)         AS servicios_activos
FROM hecho_transiciones_estado t
JOIN dim_mensajero m ON m.key_dim_mensajero = t.fk_mensajero
WHERE t.fk_mensajero != 0 AND t.fk_estado_servicio != 0
GROUP BY hora, m.nombre
ORDER BY servicios_activos DESC;
""", dw)

,hora,mensajero,servicios_activos
0,11.0,pepito_el_rapido pepito_el_furioso,7871
1,9.0,pepito_el_rapido pepito_el_furioso,7559
2,10.0,pepito_el_rapido pepito_el_furioso,7317
3,15.0,pepito_el_rapido pepito_el_furioso,6415
4,16.0,pepito_el_rapido pepito_el_furioso,5917
5,12.0,pepito_el_rapido pepito_el_furioso,5739
6,8.0,pepito_el_rapido pepito_el_furioso,5655
7,14.0,pepito_el_rapido pepito_el_furioso,5605
8,13.0,pepito_el_rapido pepito_el_furioso,4343
9,17.0,pepito_el_rapido pepito_el_furioso,4125


### P4 - Número de servicios por cliente y por mes

In [73]:
pd.read_sql("""
    SELECT c.nombre_cliente, f.month, f.nombre_mes, COUNT(*) AS servicios
    FROM hecho_servicio h
    JOIN dim_cliente c ON c.key_dim_cliente = h.fk_cliente
    JOIN dim_fecha   f ON f.key_dim_fecha   = h.fk_fecha_solicitud
    WHERE h.fk_cliente != 0 AND h.fk_fecha_solicitud != 0
    GROUP BY c.nombre_cliente, f.month, f.nombre_mes
    ORDER BY c.nombre_cliente, f.month
""", dw)

,nombre_cliente,month,nombre_mes,servicios
0,BANCO REGIONAL DE SANGRE BLOD-LIFE,2,Febrero,17
1,BANCO REGIONAL DE SANGRE BLOD-LIFE,3,Marzo,53
2,BANCO REGIONAL DE SANGRE BLOD-LIFE,4,Abril,50
3,BANCO REGIONAL DE SANGRE BLOD-LIFE,5,Mayo,7
4,BANCO REGIONAL DE SANGRE BLOD-LIFE,6,Junio,24
...,...,...,...,...
96,UNIDAD DE TRAUMA DEL OESTE,4,Abril,75
97,UNIDAD DE TRAUMA DEL OESTE,5,Mayo,106
98,UNIDAD DE TRAUMA DEL OESTE,6,Junio,102
99,UNIDAD DE TRAUMA DEL OESTE,7,Julio,97


### P5 - Mensajeros más eficientes

In [78]:
pd.read_sql("""
    SELECT m.nombre, COUNT(*) AS servicios_prestados,
           ROUND(AVG(CASE WHEN h.tiempo_total_minutos > 0.001 THEN h.tiempo_total_minutos END), 1) AS tiempo_promedio_min
    FROM hecho_servicio h
    JOIN dim_mensajero m ON m.key_dim_mensajero = h.fk_mensajero
    WHERE h.fk_mensajero != 0
    GROUP BY m.nombre
    ORDER BY servicios_prestados DESC
""", dw)

,nombre,servicios_prestados,tiempo_promedio_min
0,pepito_el_rapido pepito_el_furioso,27618,527.4


### P6 - Sedes que más servicios solicitan por cliente

In [88]:
pd.read_sql("""
    SELECT c.nombre_cliente, s.nombre_sede, s.ciudad, COUNT(*) AS total_servicios
    FROM hecho_servicio h
    JOIN dim_cliente c ON c.key_dim_cliente = h.fk_cliente
    JOIN dim_sede    s ON s.key_dim_sede    = h.fk_sede
    GROUP BY c.nombre_cliente, s.nombre_sede, s.ciudad
    ORDER BY c.nombre_cliente, total_servicios DESC
""", dw)

,nombre_cliente,nombre_sede,ciudad,total_servicios
0,BANCO REGIONAL DE SANGRE BLOD-LIFE,REMEDIOZ/ 123,CALI,134
1,BANCO REGIONAL DE SANGRE BLOD-LIFE,SIN SEDE,DESCONOCIDO,71
2,CALI SULUD Y VIDA,SIN SEDE,DESCONOCIDO,1008
3,CARROS DEL PACIFICO (CHINA),SIN SEDE,DESCONOCIDO,6235
4,CARROS DEL PACIFICO (CHINA),PASTO VITRINA /,PASTO,2687
5,CARROS DEL PACIFICO (CHINA),PASTO BODEGA 29/,PASTO,2438
6,CARROS DEL PACIFICO (CHINA),PALMIRA BODEGA 20 /,PALMIRA,1951
7,CARROS DEL PACIFICO (CHINA),CONTRIBUTIVO / ESENSA,CALI,982
8,CARROS DEL PACIFICO (CHINA),PRINCIPAL /14,CALI,642
9,CARROS DEL PACIFICO (CHINA),PROVIDA ATENCION BASICA,CALI,517


### P7 - Tiempo promedio de entrega

In [94]:
pd.read_sql("""
    SELECT
        ROUND(AVG(CASE WHEN tiempo_total_minutos > 0.001 THEN tiempo_total_minutos END)::numeric, 2) AS promedio_total_min,
        ROUND(AVG(CASE WHEN tiempo_fase_asignacion_min > 0.001 THEN tiempo_fase_asignacion_min END)::numeric, 2) AS promedio_asignacion_min,
        ROUND(AVG(CASE WHEN tiempo_fase_recogida_min > 0.001 THEN tiempo_fase_recogida_min END)::numeric, 2) AS promedio_recogida_min,
        ROUND(AVG(CASE WHEN tiempo_fase_entrega_min > 0.001 THEN tiempo_fase_entrega_min END)::numeric, 2) AS promedio_entrega_min
    FROM hecho_servicio
    WHERE tiempo_total_minutos > 0.001;
""", dw)

,promedio_total_min,promedio_asignacion_min,promedio_recogida_min,promedio_entrega_min
0,527.43,110.17,79.13,338.27


### P8 - ¿En qué fase del servicio hay más demoras?

In [85]:
pd.read_sql("""
    SELECT
        e.nombre_estado,
        e.orden_estado,
        ROUND(AVG(CASE WHEN t.duracion_en_estado_min IS NOT NULL THEN t.duracion_en_estado_min END), 2) AS duracion_promedio_min,
        COUNT(*) AS num_transiciones
    FROM hecho_transiciones_estado t
    JOIN dim_estado_servicio e ON e.key_dim_estado = t.fk_estado_servicio
    WHERE t.fk_estado_servicio != 0
    GROUP BY e.nombre_estado, e.orden_estado
    ORDER BY e.orden_estado
""", dw)

,nombre_estado,orden_estado,duracion_promedio_min,num_transiciones
0,Iniciado,1,151.70,29627
1,Con mensajero Asignado,2,95.89,30249
2,Con novedad,3,89.49,5202
3,Recogido por mensajero,4,87.22,27542
4,Entregado en destino,5,83.08,27335
5,Terminado completo,6,0.04,8447


### P9 - ¿Cuáles son las novedades que más se presentan?

In [87]:
pd.read_sql("""
    SELECT nombre_tipo_novedad, total_ocurrencias, servicios_afectados
    FROM resumen_novedades
    ORDER BY total_ocurrencias DESC
""", dw)

,nombre_tipo_novedad,total_ocurrencias,servicios_afectados
0,Novedades del servicio,5,4
1,No puedo continuar,3,2
